# 1. 날씨 자전거 전처리
## 1.1 csv불러오기

In [2]:
import pandas as pd

In [3]:
pb=pd.read_csv("Seoul_public_bicycle.csv")
pb.head()

,ID,datetime,temperature,precipitation,windspeed,humidity,dew_point,sunshine,solar_radiation,snowfall,cloud_cover,visibility,ground_temp,weekday,holiday,count
0,1,2015-09-19 00:00:00,20.2,0.0,0.8,60.0,12.1,0.0,0.00,0.0,4.0,2000.0,19.1,Saturday,0,1
1,2,2015-09-19 01:00:00,19.5,0.0,1.2,63.0,12.2,0.0,0.00,0.0,NaN,2000.0,18.3,Saturday,0,2
2,3,2015-09-19 07:00:00,17.2,0.0,2.0,75.0,12.7,0.3,0.09,0.0,3.0,2000.0,17.0,Saturday,0,1
3,4,2015-09-19 08:00:00,19.2,0.0,1.9,63.0,11.9,1.0,0.55,0.0,4.0,1800.0,19.7,Saturday,0,2
4,5,2015-09-19 09:00:00,21.1,0.0,1.0,57.0,12.2,1.0,1.12,0.0,4.0,1800.0,25.3,Saturday,0,21


## 1.2 결측값 제거

In [4]:
pb.isnull().sum()

ID                    0
datetime              0
temperature           5
precipitation      2406
windspeed           123
humidity              4
dew_point             2
sunshine           1794
solar_radiation    1796
snowfall           8967
cloud_cover        5482
visibility            1
ground_temp          32
weekday               0
holiday               0
count                 0
dtype: int64

In [5]:
pb=pb.dropna()

In [9]:
pb.isnull().sum()

ID                 0
datetime           0
temperature        0
precipitation      0
windspeed          0
humidity           0
dew_point          0
sunshine           0
solar_radiation    0
snowfall           0
cloud_cover        0
visibility         0
ground_temp        0
weekday            0
holiday            0
count              0
dtype: int64

## 1.3 열 이름 바꾸기

In [10]:
pb=pb.rename(columns={"datetime": "weather_datetime",
    "temperature": "temp",
    "precipitation": "rain",
    "windspeed": "wind",
    "humidity": "humid",
    "sunshine": "sunshine",
    "solar_radiation": "solar",
    "cloud_cover": "cloud"})
pb.columns

Index(['ID', 'weather_datetime', 'temp', 'rain', 'wind', 'humid', 'dew_point',
       'sunshine', 'solar', 'snowfall', 'cloud', 'visibility', 'ground_temp',
       'weekday', 'holiday', 'count'],
      dtype='object')

## 1.4 필요없는 열 삭제

In [11]:
cols_to_drop = [
    "ID", 
    "dew_point",
    "snowfall",
    "ground_temp",
]
pb=pb.drop(columns=cols_to_drop,errors='ignore')
pb.columns

Index(['weather_datetime', 'temp', 'rain', 'wind', 'humid', 'sunshine',
       'solar', 'cloud', 'visibility', 'weekday', 'holiday', 'count'],
      dtype='object')

## 1.5 시간 타입 변환

In [12]:
pb["weather_datetime"]=pd.to_datetime(pb["weather_datetime"], errors='coerce')

In [13]:
pb["datehour"] = pb["weather_datetime"].dt.floor("H")


C:\Users\sooyu\AppData\Local\Temp\ipykernel_2656\1559535517.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  pb["datehour"] = pb["weather_datetime"].dt.floor("H")


## 1.6 필요한 열 추가

In [14]:
# 비가 왔는지 여부
pb["rain_flag"] = (pb["rain"] > 0).astype(int)


In [15]:
pb.to_csv("pb_preprocess.csv",index=False, encoding="cp949")

## 1.7 필터링

In [17]:
weather_2024 = pb[
    (pb["weather_datetime"].dt.year == 2024) &
    (pb["weather_datetime"].dt.month.isin([9, 10]))
].copy()
weather_2024.head()

,weather_datetime,temp,rain,wind,humid,sunshine,solar,cloud,visibility,weekday,holiday,count,datehour,rain_flag
78090,2024-09-01 00:00:00,26.3,0.0,1.5,71.0,0.0,0.0,0.0,1917.0,Sunday,0,3895,2024-09-01 00:00:00,0
78091,2024-09-01 01:00:00,25.7,0.0,1.3,74.0,0.0,0.0,0.0,1848.0,Sunday,0,2584,2024-09-01 01:00:00,0
78092,2024-09-01 02:00:00,25.3,0.0,0.5,77.0,0.0,0.0,0.0,1613.0,Sunday,0,1739,2024-09-01 02:00:00,0
78093,2024-09-01 03:00:00,25.1,0.0,0.7,81.0,0.0,0.0,3.0,1421.0,Sunday,0,1249,2024-09-01 03:00:00,0
78094,2024-09-01 04:00:00,24.8,0.0,0.2,79.0,0.0,0.0,8.0,1445.0,Sunday,0,878,2024-09-01 04:00:00,0


In [18]:
weather_2024.to_csv("weather_bike_preprocess.csv",index=False, encoding="cp949")

# 2. 대여소 정보 전처리
## 2.1 csv 불러오기

In [ ]:
import pandas as pd

In [ ]:
df9=pd.read_csv("서울특별시 공공자전거 대여이력 정보_2409.csv",encoding='cp949', encoding_errors='ignore',na_values=["\\N", "\\\\N", "N", ""],keep_default_na=True    )
df10=pd.read_csv("서울특별시 공공자전거 대여이력 정보_2410.csv",encoding='cp949', encoding_errors='ignore',na_values=["\\N", "\\\\N", "N", ""],keep_default_na=True    )

In [ ]:
df9.columns

Index(['자전거번호', '대여일시', '대여 대여소번호', '대여 대여소명', '대여거치대', '반납일시', '반납대여소번호',
       '반납대여소명', '반납거치대', '이용시간(분)', '이용거리(M)', '생년', '성별', '이용자종류', '대여대여소ID',
       '반납대여소ID', '자전거구분'],
      dtype='object')

In [ ]:
df9.shape
df10.shape

(4686675, 17)

## 2.2 이용거리 / 이용시간 0 제거

In [ ]:
df9 = df9[(df9["이용시간(분)"] > 0) & (df9["이용거리(M)"] > 0)]
df10 = df10[(df10["이용시간(분)"] > 0) & (df10["이용거리(M)"] > 0)]


## 2.3 결측값 제거

In [ ]:
df9.isnull().sum()

자전거번호             0
대여일시              0
대여 대여소번호      28042
대여 대여소명       36572
대여거치대         36572
반납일시              0
반납대여소번호       43391
반납대여소명        51674
반납거치대         54252
이용시간(분)           0
이용거리(M)           0
생년           574019
성별          1411967
이용자종류        318218
대여대여소ID           0
반납대여소ID       15502
자전거구분             0
dtype: int64

In [ ]:
df10.isnull().sum()

자전거번호          7855
대여일시              0
대여 대여소번호      52229
대여 대여소명       61957
대여거치대         61957
반납일시              0
반납대여소번호       69146
반납대여소명        78738
반납거치대         82500
이용시간(분)           0
이용거리(M)           0
생년           687360
성별          1591739
이용자종류        410734
대여대여소ID           0
반납대여소ID       16992
자전거구분          7855
dtype: int64

In [ ]:
df9=df9.dropna()
df10=df10.dropna()

## 2.4 샘플링

In [ ]:
sample9 = df9.sample(frac=0.005, random_state=42)
sample10 = df10.sample(frac=0.005, random_state=42)

## 2.5 두달 데이터 합치기

In [ ]:
bike = pd.concat([sample9, sample10], ignore_index=True)

In [ ]:
bike.shape

(25753, 17)

In [ ]:
bike.to_csv("bike_sampled.csv", index=False, encoding="cp949")

## 2.6 열 이름 변경

In [ ]:
bike_sample=pd.read_csv("bike_sampled.csv", encoding='cp949', encoding_errors='ignore')

In [ ]:
bike_sample.columns

Index(['자전거번호', '대여일시', '대여 대여소번호', '대여 대여소명', '대여거치대', '반납일시', '반납대여소번호',
       '반납대여소명', '반납거치대', '이용시간(분)', '이용거리(M)', '생년', '성별', '이용자종류', '대여대여소ID',
       '반납대여소ID', '자전거구분'],
      dtype='object')

In [ ]:
bike_sample = bike_sample.rename(columns={
    "자전거번호": "bike_id",
    "대여일시": "rent_datetime",
    "대여 대여소번호": "rent_station_id",
    "대여 대여소명": "rent_station_name",
    "대여거치대": "rent_rack",

    "반납일시": "return_datetime",
    "반납대여소번호": "return_station_id",
    "반납대여소명": "return_station_name",
    "반납거치대": "return_rack",

    "이용시간": "use_time",
    "이용거리": "use_distance",

    "생년": "birth_year",
    "성별": "gender",
    "이용자종류": "user_type",

    "이용시간(분)": "use_time_min",
    "이용거리(M)": "use_distance_m",
    "대여대여소ID": "rent_station_uid",
    "반납대여소ID": "return_station_uid",
    "자전거구분": "bike_type",
})

In [ ]:
bike_sample.columns

Index(['bike_id', 'rent_datetime', 'rent_station_id', 'rent_station_name',
       'rent_rack', 'return_datetime', 'return_station_id',
       'return_station_name', 'return_rack', 'use_time_min', 'use_distance_m',
       'birth_year', 'gender', 'user_type', 'rent_station_uid',
       'return_station_uid', 'bike_type'],
      dtype='object')

## 2.7 필요 없는 열 삭제

In [ ]:
columns_to_drop = [
    "rent_rack", "return_rack",
    "bike_id",
]

bike_sample = bike_sample.drop(columns=columns_to_drop, errors="ignore")

## 2.8 시간 타입 변환

In [ ]:
bike_sample["rent_datetime"] = pd.to_datetime(bike_sample["rent_datetime"], errors='coerce')
bike_sample["return_datetime"] = pd.to_datetime(bike_sample["return_datetime"], errors='coerce')

In [ ]:
bike_sample["datehour"] = bike_sample["rent_datetime"].dt.floor("H")


C:\Users\sooyu\AppData\Local\Temp\ipykernel_20976\1843226611.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  bike_sample["datehour"] = bike_sample["rent_datetime"].dt.floor("H")


## 2.9 필요한 열 추가

In [ ]:
# 반납일시 - 대여일시 = 실제 사용시간 계산 
bike_sample["actual_use_time"] = (bike_sample["return_datetime"] - bike_sample["rent_datetime"]).dt.total_seconds()

In [ ]:
# 요일별, 주말/평일 컬럼
bike_sample["weekday"] = bike_sample["rent_datetime"].dt.weekday   # 0~6 (월~일)
bike_sample["weekday_name"] = bike_sample["rent_datetime"].dt.day_name()

In [ ]:
bike_sample.to_csv("bike_sampled_preprocess.csv", index=False, encoding="cp949")